In [11]:
import sys
import pandas as pd
sys.path.insert(0, '.')

from src.validators import (
    validate_completeness,
    validate_known_counterparties,
    validate_date_logic,
    validate_positive_notional,
    validate_counterparty_limits,
    validate_emir_reporting,
    validate_no_duplicates
)
from src.report_generator import generate_report, save_report

print("Libraries loaded. Pipeline ready.")

Libraries loaded. Pipeline ready.


In [12]:
transactions = pd.read_csv('./data/raw_transactions.csv')
limits = pd.read_csv('./data/counterparty_limits.csv')

print(f"Transactions loaded: {transactions.shape[0]} rows, {transactions.shape[1]} columns")
print(f"Counterparties loaded: {limits.shape[0]} rows")
print(f"\nColumns: {list(transactions.columns)}")

Transactions loaded: 103 rows, 16 columns
Counterparties loaded: 5 rows

Columns: ['trade_id', 'trade_date', 'value_date', 'maturity_date', 'instrument_type', 'base_currency', 'quote_currency', 'notional_amount', 'notional_currency', 'counterparty_id', 'buy_sell', 'status', 'trade_repository_ref', 'clearing_eligible', 'booking_timestamp', 'source_system']


In [13]:
print("=== DUPLICATE TRADE IDs ===")
dupes = transactions[transactions.duplicated('trade_id', keep=False)]
print(dupes[['trade_id', 'booking_timestamp', 'status']].sort_values('trade_id'))

print("\n=== NULL COUNTERPARTY IDs ===")
nulls = transactions[transactions['counterparty_id'].isna()]
print(nulls[['trade_id', 'counterparty_id', 'instrument_type']])

print("\n=== INVALID INSTRUMENT TYPES ===")
valid_types = {'FX_FORWARD', 'INTEREST_RATE_SWAP', 'CASH_DEPOSIT', 'COMMODITY_FORWARD'}
invalid = transactions[~transactions['instrument_type'].isin(valid_types)]
print(invalid[['trade_id', 'instrument_type']])

=== DUPLICATE TRADE IDs ===
     trade_id    booking_timestamp   status
10   TRD-0011  2024-06-23T00:00:00  PENDING
100  TRD-0011  2024-06-23T02:00:00  PENDING
25   TRD-0026  2024-04-19T00:00:00  PENDING
101  TRD-0026  2024-04-19T02:00:00  PENDING
40   TRD-0041  2024-11-18T00:00:00   ACTIVE
102  TRD-0041  2024-11-18T02:00:00   ACTIVE

=== NULL COUNTERPARTY IDs ===
    trade_id counterparty_id    instrument_type
15  TRD-0016             NaN         FX_FORWARD
30  TRD-0031             NaN  COMMODITY_FORWARD
50  TRD-0051             NaN  COMMODITY_FORWARD

=== INVALID INSTRUMENT TYPES ===
    trade_id instrument_type
3   TRD-0004    FX_FORWARD  
8   TRD-0009         FX_SPOT
22  TRD-0023            BOND


In [14]:
# Stage 2: Clean
df_clean = transactions.copy()

string_cols = ['trade_id', 'instrument_type', 'counterparty_id', 'buy_sell', 'status']
for col in string_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()
        df_clean[col] = df_clean[col].replace('nan', pd.NA)

for col in ['trade_date', 'value_date', 'maturity_date']:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce').dt.strftime('%Y-%m-%d')

df_clean['booking_timestamp'] = pd.to_datetime(df_clean['booking_timestamp'], errors='coerce')
df_clean = df_clean.sort_values('booking_timestamp', ascending=False)
df_clean = df_clean.drop_duplicates('trade_id', keep='first')
df_clean = df_clean[df_clean['instrument_type'].isin(valid_types)]

print(f"After cleaning: {len(df_clean)} records (from {len(transactions)})")

# Stage 3: Validate
results = [
    validate_completeness(df_clean),
    validate_known_counterparties(df_clean, limits),
    validate_date_logic(df_clean),
    validate_positive_notional(df_clean),
    validate_counterparty_limits(df_clean, limits),
    validate_emir_reporting(df_clean),
    validate_no_duplicates(df_clean),
]

print("Validation complete.")

After cleaning: 98 records (from 103)
Validation complete.


In [15]:
passed = [r for r in results if r['status'] == 'PASS']
failed = [r for r in results if r['status'] == 'FAIL']
overall = 'PASS' if not failed else 'FAIL'

print(f"OVERALL STATUS: {overall}")
print(f"Rules passed: {len(passed)} | Rules failed: {len(failed)}\n")

for r in results:
    icon = "✅" if r['status'] == 'PASS' else "❌"
    print(f"{icon} {r['rule_id']:8} | {r['category']:20} | Passed: {r['passed']} | Failed: {r['failed']}")

OVERALL STATUS: FAIL
Rules passed: 2 | Rules failed: 5

❌ V1       | COMPLETENESS         | Passed: 95 | Failed: 3
❌ V3       | COMPLETENESS         | Passed: 95 | Failed: 3
❌ V4_V5    | FINANCIAL_LOGIC      | Passed: 95 | Failed: 3
❌ V6       | FINANCIAL_LOGIC      | Passed: 97 | Failed: 1
✅ V10      | LIMIT_MONITORING     | Passed: 5 | Failed: 0
❌ V13      | REGULATORY           | Passed: 13 | Failed: 5
✅ V15      | REGULATORY           | Passed: 98 | Failed: 0


In [16]:
print("=== V13: EMIR REPORTING GAPS ===")
emir = next(r for r in results if r['rule_id'] == 'V13')
print(f"Active trades missing trade_repository_ref: {emir['failed']}")
for rec in emir['failing_records']:
    print(f"  {rec['trade_id']} | {rec['instrument_type']} | {rec['trade_date']}")

print("\n=== V4_V5: DATE LOGIC FAILURES ===")
dates = next(r for r in results if r['rule_id'] == 'V4_V5')
print(f"Records with impossible dates: {dates['failed']}")
for rec in dates['failing_records']:
    print(f"  {rec['trade_id']} | trade: {rec['trade_date']} | value: {rec['value_date']} | maturity: {rec['maturity_date']}")

print("\nBusiness impact: EMIR gaps = regulatory breach. Date errors = incorrect settlement and risk calculations.")

=== V13: EMIR REPORTING GAPS ===
Active trades missing trade_repository_ref: 5
  TRD-0018 | FX_FORWARD | 2024-12-03
  TRD-0034 | CASH_DEPOSIT | 2024-05-22
  TRD-0007 | INTEREST_RATE_SWAP | 2024-05-18
  TRD-0033 | INTEREST_RATE_SWAP | 2024-04-12
  TRD-0004 | FX_FORWARD | 2024-02-19

=== V4_V5: DATE LOGIC FAILURES ===
Records with impossible dates: 3
  TRD-0008 | trade: 2024-10-17 00:00:00 | value: 2024-10-14 00:00:00 | maturity: 2025-03-08 00:00:00
  TRD-0021 | trade: 2024-07-24 00:00:00 | value: 2024-07-21 00:00:00 | maturity: 2024-11-17 00:00:00
  TRD-0036 | trade: 2024-12-28 00:00:00 | value: 2025-01-02 00:00:00 | maturity: 2024-12-23 00:00:00

Business impact: EMIR gaps = regulatory breach. Date errors = incorrect settlement and risk calculations.


In [17]:
report = generate_report(results)
save_report(report, './outputs/validation_report.json')

print(f"\nRecommendation: {report['recommendation']}")

Report saved to: ./outputs/validation_report.json

Recommendation: Data NOT approved. 5 rule(s) failed. Review failures before use.
